In [ ]:
# Import necessary libraries
from pyspark.sql.functions import col, current_date
from delta.tables import DeltaTable
from com.microsoft.spark.fabric.Constants import Constants
import traceback

StatementMeta(, cbc1ea06-f308-4994-bd35-9b54f4d7c50c, 3, Finished, Available, Finished)

#### Create Temporary Views of Warehouse Tables

In [ ]:
# Query to get warehouse tables and their columns from the configuration
warehouse_tables_sql = """
    SELECT
        d.source_name                                   AS source_name,
        de.storage_name                                 AS storage_name,
        de.schema_name                                  AS schema_name,
        de.table_name                                   AS table_name,
        CONCAT_WS(', ', COLLECT_SET(rb.column_name))    AS column_names
    FROM
        lh_utils.dq_config.data_entity      AS de
    INNER JOIN
        lh_utils.dq_config.rule_binding     AS rb
            ON de.entity_id     = rb.entity_id
            AND rb.is_active    = 1
    INNER JOIN
        lh_utils.dq_config.dataset          AS d
            ON de.dataset_id    = d.id
    WHERE
        LOWER(de.storage_type) = 'warehouse'
    GROUP BY
        d.source_name,
        de.storage_name,
        de.schema_name,
        de.table_name
"""
warehouse_tables_df = spark.sql(warehouse_tables_sql)

# Iterate through each warehouse table
for row in warehouse_tables_df.collect():
    storage_name = row.storage_name
    schema_name = row.schema_name
    table_name = row.table_name
    column_names = row.column_names if row.column_names else "*"  # Use all columns if none specified
    view_name = f"{storage_name}_{schema_name}_{table_name}_view"
    
    # Read using Synapse SQL
    try:
        spark.read \
            .option(Constants.DatabaseName, storage_name) \
            .synapsesql(f"""
                SELECT
                    {column_names}
                FROM
                    {schema_name}.{table_name}
            """).createOrReplaceTempView(view_name)
        print(f"Successfully created view: {view_name}")
    
    except Exception as e:
        print(f"Error processing {schema_name}.{table_name}: {str(e)}")
        traceback.print_exc()

StatementMeta(, cbc1ea06-f308-4994-bd35-9b54f4d7c50c, 4, Finished, Available, Finished)

Successfully created view: wh_curated_report_ev_charging_stations_rpt_charging_stations_view
Successfully created view: wh_curated_report_ev_charging_stations_rpt_world_summary_view


In [ ]:
# Prepare to clean up existing error counts for the current refresh date
workspace_name  = "ws-data-quality"
lh_name         = "lh_utils"
output_schema   = "data_governance"
output_table    = "data_quality_error_counts"

output_path = f"abfss://{workspace_name}@onelake.dfs.fabric.microsoft.com/{lh_name}.Lakehouse/Tables/{output_schema}/{output_table}"

if DeltaTable.isDeltaTable(spark, output_path):
    # Get the maximum refreshed_at date from the data_quality_errors table
    max_date_df = spark.sql("SELECT MAX(DATE(refreshed_at)) as max_date FROM lh_utils.data_governance.data_quality_errors")
    max_date = max_date_df.collect()[0]['max_date']
    
    spark.sql(f"DELETE FROM {lh_name}.{output_schema}.{output_table} WHERE date_refreshed = '{max_date}'")

# Query to get data entities and their metadata
tables_df = spark.sql("""
    SELECT
        ds.source                                     AS source_name
        , de.entity_id
        , de.storage_name
        , LOWER(de.storage_type)                      AS storage_type
        , de.schema_name
        , de.table_name
        , de.table_display_name
        , COALESCE(de.data_owner   ,ds.data_owner)    AS data_owner
        , COALESCE(de.data_steward ,ds.data_steward)  AS data_steward     
    FROM
        lh_utils.dq_config.data_entity  AS de
    INNER JOIN
        lh_utils.dq_config.dataset      AS ds
            ON de.dataset_id = ds.id
""")

display(tables_df)

StatementMeta(, cbc1ea06-f308-4994-bd35-9b54f4d7c50c, 5, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 17792e5f-3f45-43b9-a6e2-d0c24fb713b0)

In [4]:
# Collect the table metadata
tables_metadata = tables_df.select(
    col("source_name").alias("source_display_name"),
    col("storage_name"),
    col("storage_type"),
    col("schema_name"),
    col("table_name"), 
    col("table_display_name"),
    col("data_owner"),
    col("data_steward"),
).collect()

# Define special conditions for specific tables
special_conditions = {
    'job_revenue': "AND sold_by_id IS NOT NULL",
    'stages': "",
}


# Generate the dynamic query parts
query_parts = []
for row in tables_metadata:
    query_part = f"""
        SELECT
            '{row.table_display_name}'  AS table_name,
            '{row.source_display_name}' AS source,
            '{row.data_owner}'          AS data_owner,
            '{row.data_steward}'        AS data_steward,
            COUNT(*)                    AS row_count
        FROM
            {f'{row.storage_name}.{row.schema_name}.{row.table_name}' if row.storage_type == "lakehouse" else f'{row.storage_name}_{row.schema_name}_{row.table_name}_view'}
    """
    query_parts.append(query_part.strip())

# Combine all parts with UNION ALL
final_query = "\nUNION ALL\n".join(query_parts)
# print(final_query)

# Create temp view
spark.sql(final_query).createOrReplaceTempView("row_counts")

StatementMeta(, cbc1ea06-f308-4994-bd35-9b54f4d7c50c, 6, Finished, Available, Finished)

In [ ]:
final_error_counts_sql = """
    WITH error_counts AS (
        SELECT
            source
            , table_name
            , COUNT(*)                    AS error_count
            , dq_metric
            , DATE(refreshed_at)          AS date_refreshed
        FROM
            lh_utils.data_governance.data_quality_errors
        WHERE
            1 = 1
        GROUP BY
            table_name,
            source,
            dq_metric,
            CAST(refreshed_at AS DATE)
    )

    SELECT
        ec.source
        , ec.table_name
        , ec.error_count
        , rc.row_count
        , ec.dq_metric
        , rc.data_owner
        , rc.data_steward
        , ec.date_refreshed
    FROM
        error_counts AS ec
    LEFT JOIN
        row_counts   AS rc
            ON  ec.table_name   = rc.table_name
            AND ec.source       = rc.source
"""

# Write the final error counts to the Delta table
spark.sql(final_error_counts_sql).write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save(output_path)

# print(spark.sql(final_error_counts_sql).count())

StatementMeta(, cbc1ea06-f308-4994-bd35-9b54f4d7c50c, 7, Finished, Available, Finished)

In [6]:
# mssparkutils.session.stop()

StatementMeta(, cbc1ea06-f308-4994-bd35-9b54f4d7c50c, 8, Finished, Available, Finished)